# 3.0. Load Silver Table

In [0]:
# Load Silver table
from pyspark.sql.functions import col

df_silver = spark.table("silver_operational_reporting")
df_silver.show(5)


+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+----------------------+--------------------------+---------------------+
|Request_ID|Department|         Start_Time|           End_Time|SLA_Status|Processing_Time|Employee_ID|Priority|Invalid_Timestamp_Flag|Processing_Time_Recomputed|Invalid_Employee_Flag|
+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+----------------------+--------------------------+---------------------+
|  REQ00001|   Finance|2026-06-14 03:42:00|               NULL|  Breached|           -1.0|       E353|     Low|                     1|                      -1.0|                    1|
|  REQ00002|Operations|2026-06-05 14:23:00|               NULL|  Breached|           -1.0|       E424|     Low|                     1|                      -1.0|                    1|
|  REQ00003|        IT|2026-06-13 12:46:00|2026-06-13 19:32:00|  Breached|      

### Verify Schema

In [0]:
df_silver.printSchema()


root
 |-- Request_ID: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- SLA_Status: string (nullable = true)
 |-- Processing_Time: double (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- Invalid_Timestamp_Flag: integer (nullable = true)
 |-- Processing_Time_Recomputed: double (nullable = true)
 |-- Invalid_Employee_Flag: integer (nullable = true)



### Row Count Audit

In [0]:
print("Silver row count:", df_silver.count())


Silver row count: 10000


# 3.1: Deduplication

### Drop Duplicate Records
We’ll use Request_ID as the unique identifier. Any duplicates must be removed.

In [0]:
# Deduplicate based on Request_ID
df_silver_dedup = df_silver.dropDuplicates(["Request_ID"])


### Audit Logging
Compare row counts before and after deduplication to ensure transparency.

In [0]:
silver_count_before = df_silver.count()
silver_count_after = df_silver_dedup.count()

print("Rows before deduplication:", silver_count_before)
print("Rows after deduplication:", silver_count_after)
print("Rows removed as duplicates:", silver_count_before - silver_count_after)


Rows before deduplication: 10000
Rows after deduplication: 10000
Rows removed as duplicates: 0


# 3.2: Fact Table Creation

### Select Core Columns
The fact table should contain identifiers, timestamps, metrics, and status fields:

In [0]:
fact_requests = df_silver_dedup.select(
    "Request_ID",
    "Department",
    "Employee_ID",
    "Start_Time",
    "End_Time",
    "Processing_Time_Recomputed",
    "SLA_Status",
    "Priority"
)


### Persist Fact Table
Save the fact table into Delta Lake for downstream queries:

In [0]:
fact_requests.write.format("delta").mode("overwrite").saveAsTable("fact_requests")


### Audit Logging
Confirm row counts match the deduplicated Silver dataset:

In [0]:
fact_count = fact_requests.count()
print("Fact table row count:", fact_count)


Fact table row count: 10000


# 3.3: Dimension Tables

### Department Dimension
Contains unique department values.\
Used to slice KPIs by department.

In [0]:
dim_department = df_silver_dedup.select("Department").distinct()
dim_department.write.format("delta").mode("overwrite").saveAsTable("dim_department")


### Employee Dimension
Contains unique employee IDs.\
Ensures referential integrity for workforce analytics.

In [0]:
dim_employee = df_silver_dedup.select("Employee_ID").distinct()
dim_employee.write.format("delta").mode("overwrite").saveAsTable("dim_employee")


### Priority Dimension
Contains unique priority values.\
Enables SLA breach analysis by priority level.

In [0]:
dim_priority = df_silver_dedup.select("Priority").distinct()
dim_priority.write.format("delta").mode("overwrite").saveAsTable("dim_priority")


### Audit Logging
Always confirm row counts for each dimension:

In [0]:
print("Department dimension count:", dim_department.count())
print("Employee dimension count:", dim_employee.count())
print("Priority dimension count:", dim_priority.count())


Department dimension count: 4
Employee dimension count: 401
Priority dimension count: 3
